In [18]:
import os
import shutil
import re
import pandas as pd
import numpy as np
import glob
from pathlib import Path
from tqdm.auto import tqdm

# 1. CONFIGURATION & PATHS

In [19]:
# Input Paths (Adjust if your Kaggle dataset name is different)
RAW_DATA_DIR = Path("/kaggle/input/mlops-torob-dataset")
# This is where your downloaded images are currently sitting
IMAGE_SOURCE_DIR = Path("/kaggle/input/torob-images/balanced_images") 

# Output Paths
OUTPUT_ROOT = Path("/kaggle/working/mytorobdataset")
OUTPUT_IMAGES = OUTPUT_ROOT / "images"
OUTPUT_DATA_FILE = OUTPUT_ROOT / "processed_metadata.parquet"

# Create directories
OUTPUT_IMAGES.mkdir(parents=True, exist_ok=True)

print(f"📂 Configuration Set:")
print(f"   - Raw Data:      {RAW_DATA_DIR}")
print(f"   - Image Source:  {IMAGE_SOURCE_DIR}")
print(f"   - Final Output:  {OUTPUT_ROOT}")


📂 Configuration Set:
   - Raw Data:      /kaggle/input/mlops-torob-dataset
   - Image Source:  /kaggle/input/torob-images/balanced_images
   - Final Output:  /kaggle/working/mytorobdataset


# 2. ADVANCED NORMALIZATION (DIACRITICS + ENGLISH NUMBERS)

In [20]:
def clean_persian_text(text):
    """
    Converts Persian/Arabic numbers to English, removes diacritics, 
    removes punctuation, and standardizes characters.
    """
    if not isinstance(text, str): return ""
    
    # 1. Map Persian/Arabic Digits to English
    persian_digits = "۰۱۲۳۴۵۶۷۸۹"
    arabic_digits = "٠١٢٣٤٥٦٧٨٩"
    english_digits = "0123456789"
    
    # Translation table
    trans_table = str.maketrans(persian_digits + arabic_digits, english_digits * 2)
    text = text.translate(trans_table)
    
    # 2. Fix Number Formatting (Remove commas inside numbers: 12,000 -> 12000)
    text = re.sub(r'(\d),(\d{3})', r'\1\2', text)
    
    # 3. Remove Diacritics (Fatha, Kasra, etc.)
    text = re.sub(r'[\u064B-\u0652]', '', text)
    
    # 4. Standardize Characters (Ye/Ke/ZWNJ)
    text = text.replace('ي', 'ی').replace('ك', 'ک').replace('ى', 'ی')
    text = text.replace('\u200c', ' ') # ZWNJ -> Space
    
    # 5. Remove Special Symbols (Keep only word chars, spaces)
    # \w includes 0-9 and Persian letters
    text = re.sub(r'[^\w\s]', ' ', text)
    
    # 6. Collapse Whitespace
    return re.sub(r'\s+', ' ', text).strip()

# Test the function
sample_text = "سلام! من کِتابِ «جنگ و صلح» را دوست دارم. (قیمت: ۱۲,۰۰۰ تومان)"
print(f"\n🧪 Normalization Test:")
print(f"   Original: {sample_text}")
print(f"   Cleaned:  {clean_persian_text(sample_text)}")


🧪 Normalization Test:
   Original: سلام! من کِتابِ «جنگ و صلح» را دوست دارم. (قیمت: ۱۲,۰۰۰ تومان)
   Cleaned:  سلام من کتاب جنگ و صلح را دوست دارم قیمت 12000 تومان


# 3. SYNCHRONIZATION: IMAGES <-> METADATA

In [21]:
print("\n🔄 Synchronizing Data...")

# A. Scan Images
image_files = list(IMAGE_SOURCE_DIR.glob("*.jpg"))
available_keys = {f.stem for f in image_files}
print(f"   - Found {len(available_keys):,} images.")

# B. Load Metadata
print("   - Loading Parquet files...")
df_products = pd.read_parquet(RAW_DATA_DIR / "base_products.parquet")
df_cats = pd.read_parquet(RAW_DATA_DIR / "categories.parquet")
df_members = pd.read_parquet(RAW_DATA_DIR / "members.parquet")

# C. Calculate Mean Price
price_stats = df_members.groupby('base_random_key')['price'].mean().reset_index()

# D. Merge & Filter
full_df = df_products.merge(price_stats, left_on='random_key', right_on='base_random_key', how='left')
full_df = full_df.merge(df_cats[['id', 'title']], left_on='category_id', right_on='id', how='left')
full_df.rename(columns={'title': 'category_name', 'mean': 'price'}, inplace=True)

filtered_df = full_df[full_df['random_key'].isin(available_keys)].copy()
print(f"   - Matched {len(filtered_df):,} rows.")

# --- LIVE VERIFICATION MECHANISM ---
print("\n👁️  NORMALIZATION VERIFICATION (First 5 Rows):")
print("="*60)
print(f"{'ORIGINAL TEXT':<50} | {'CLEANED TEXT'}")
print("-" * 60)

# Check the first 5 valid names
sample_rows = filtered_df[['persian_name']].head(5)
for raw_text in sample_rows['persian_name']:
    cleaned = clean_persian_text(raw_text)
    # Truncate for display if too long
    print(f"{raw_text[:48]:<50} | {cleaned}")
    
# Manual Test for Price Logic
print("-" * 60)
test_str = "گوشی ۱۲,۰۰۰ تومانی (مدل جدید!)"
print(f"{test_str:<50} | {clean_persian_text(test_str)}  <-- (Manual Test)")
print("="*60)


🔄 Synchronizing Data...
   - Found 9,151 images.
   - Loading Parquet files...
   - Matched 9,151 rows.

👁️  NORMALIZATION VERIFICATION (First 5 Rows):
ORIGINAL TEXT                                      | CLEANED TEXT
------------------------------------------------------------
تابلو اسب های کاسپین (58*43)                       | تابلو اسب های کاسپین 58 43
ست جارو خاک انداز لیمون مدل دسته بلند کد IL1240    | ست جارو خاک انداز لیمون مدل دسته بلند کد IL1240 خاکستری
کتری قوری استیل شیردار کد 9050 (پست پیشتاز پس کر   | کتری قوری استیل شیردار کد 9050 پست پیشتاز پس کرایه
گیپور سفید, مشکی طرح گل – کد JIB-256               | گیپور سفید مشکی طرح گل کد JIB 256
بشقاب سرامیکی طرح نیایش                            | بشقاب سرامیکی طرح نیایش
------------------------------------------------------------
گوشی ۱۲,۰۰۰ تومانی (مدل جدید!)                     | گوشی 12000 تومانی مدل جدید  <-- (Manual Test)


# 4. APPLY CLEANING & SAVE

In [22]:
print("\n🧽 Applying Normalization to Dataset...")

# Apply cleaning to relevant text columns
filtered_df['clean_name'] = filtered_df['persian_name'].apply(clean_persian_text)
filtered_df['clean_category'] = filtered_df['category_name'].fillna("").apply(clean_persian_text)

# Create the 'unified_text' field (Crucial for RAG)
# We combine Name + Category + Brand (if available) into one clean string
filtered_df['unified_text'] = (
    filtered_df['clean_name'] + " " + 
    filtered_df['clean_category']
)

# Add relative image path for the dataset
# This is how the model will find the image: "images/xyz.jpg"
filtered_df['image_path'] = filtered_df['random_key'].apply(lambda x: f"images/{x}.jpg")

# Select final schema columns
final_cols = ['random_key', 'image_path', 'clean_name', 'unified_text', 'price', 'category_name']
final_dataset = filtered_df[final_cols].copy()

# Save Metadata
final_dataset.to_parquet(OUTPUT_DATA_FILE, index=False)
print(f"   ✅ Metadata saved to: {OUTPUT_DATA_FILE}")


🧽 Applying Normalization to Dataset...
   ✅ Metadata saved to: /kaggle/working/mytorobdataset/processed_metadata.parquet


# 5. FINALIZE IMAGE FOLDER

In [23]:
print("\n📦 Packaging Images...")

# Since we are creating a clean dataset, we want the images inside the 'mytorobdataset/images' folder.
# We can either Move them or Copy them. Copy is safer in Notebooks so you don't lose the originals if you rerun.

copied_count = 0
for img_path in tqdm(image_files, desc="Copying Images"):
    dest_path = OUTPUT_IMAGES / img_path.name
    if not dest_path.exists():
        shutil.copy2(img_path, dest_path)
    copied_count += 1

print("\n" + "="*50)
print(f"🎉 FINAL DATASET READY: {OUTPUT_ROOT}")
print("="*50)
print(f"Structure:")
print(f"   ├── processed_metadata.parquet  ({len(final_dataset)} rows)")
print(f"   └── images/                     ({copied_count} images)")
print("-" * 50)
print("\nNext Step: You can now point your DataLoader to this folder for embedding generation.")
print("You can now zip this folder or use it directly for RAG training.")


📦 Packaging Images...


Copying Images:   0%|          | 0/9151 [00:00<?, ?it/s]


🎉 FINAL DATASET READY: /kaggle/working/mytorobdataset
Structure:
   ├── processed_metadata.parquet  (9151 rows)
   └── images/                     (9151 images)
--------------------------------------------------

Next Step: You can now point your DataLoader to this folder for embedding generation.
You can now zip this folder or use it directly for RAG training.
